# Minimal GRAPE test

This notebook tests `grape.py`, the simplified no-class implementation.

The model is intentionally small and readable: 80 pulse coefficients, built from 4 real channels times 20 quadratic B-splines. The pulse starts and ends at zero because the first two and last two B-splines are skipped.


In [ ]:
from pathlib import Path
import sys

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

# This works whether Jupyter was started from the repo root or from this folder.
cwd = Path.cwd()
project_dir = cwd if (cwd / 'grape.py').exists() else cwd / 'Simplified_adaptive_grape'
repo_root = project_dir.parent
sys.path.insert(0, str(project_dir))
sys.path.insert(0, str(repo_root))

import grape


## Build the Hamiltonian and pulse basis

For quick local tests you can reduce `n_cav` or `n_iter`. For the more realistic cavity truncation, use `n_cav = 25`.


In [ ]:
config_path = repo_root / 'configuration.json'

physics = grape.load_physics_from_json(config_path, mu_qub=20.0, mu_cav=20.0)

settings = grape.make_settings()
settings['n_cav'] = 25
settings['target_n'] = 2
settings['t_drive'] = 1.408
settings['n_time_steps'] = 80
settings['n_iter'] = 300
settings['learning_rate'] = 0.03

system = grape.make_system(settings)

print('chi [rad/us]:', float(physics['chi']))
print('self-Kerr [rad/us]:', float(physics['self_kerr']))
print('target Fock state:', settings['target_n'])
print('Hilbert dimension:', 2 * settings['n_cav'])
print('number of coefficients:', 4 * settings['n_bspline'])
print('basis endpoint max:', float(jnp.max(jnp.abs(system['basis_edges'][:, [0, -1]]))))
print('max active B-splines:', int(jnp.max(jnp.sum(system['basis_mid'] > 1e-12, axis=0))))


In [ ]:
plt.figure(figsize=(9, 3))
for b in system['basis_edges']:
    plt.plot(system['t_edges'], b, lw=1)
plt.title('20 quadratic B-splines after skipping the first/last two')
plt.xlabel('time [us]')
plt.ylabel('basis value')
plt.grid(alpha=0.3)
plt.show()


## Run plain GRAPE

This is just differentiating the simulated final Fock probability with respect to the 80 bounded pulse coefficients. No measurements, no adaptive calibration, no residual model.


In [ ]:
key = jax.random.key(0)
initial_coefficients = grape.random_coefficients(key, settings, scale=0.03)
initial_probability = grape.fock_probability(initial_coefficients, system, physics, settings)
print('initial P_n:', float(initial_probability))

result = grape.run_grape(system, physics, settings, key, initial_coefficients)
coefficients = result['coefficients']
history = result['history']

print('final P_n:', float(result['final_probability']))
print('max abs coefficient:', float(jnp.max(jnp.abs(coefficients))))


In [ ]:
probability = history[:, 0]
regularization = history[:, 1]
grad_norm = history[:, 3]

fig, axes = plt.subplots(1, 3, figsize=(13, 3.2), constrained_layout=True)
axes[0].plot(probability)
axes[0].set_title('target probability')
axes[0].set_xlabel('GRAPE step')
axes[0].set_ylabel('P_n')
axes[0].grid(alpha=0.3)

axes[1].plot(-jnp.log10(jnp.maximum(1.0 - probability, 1e-8)))
axes[1].set_title('log infidelity')
axes[1].set_xlabel('GRAPE step')
axes[1].set_ylabel('-log10(1 - P_n)')
axes[1].grid(alpha=0.3)

axes[2].semilogy(jnp.maximum(grad_norm, 1e-12), label='grad norm')
axes[2].semilogy(jnp.maximum(regularization, 1e-12), label='regularization')
axes[2].set_title('optimizer diagnostics')
axes[2].set_xlabel('GRAPE step')
axes[2].legend()
axes[2].grid(alpha=0.3)
plt.show()


In [ ]:
qubit_drive, cavity_drive = grape.pulse_fields(coefficients, system, settings)
coeffs = coefficients.reshape(4, settings['n_bspline'])
edge_channels = coeffs @ system['basis_edges']

print('channel values at first time point:', edge_channels[:, 0])
print('channel values at last time point:', edge_channels[:, -1])

fig, axes = plt.subplots(1, 2, figsize=(11, 3.2), constrained_layout=True)
axes[0].plot(system['t_mid'], qubit_drive.real, label='I')
axes[0].plot(system['t_mid'], qubit_drive.imag, label='Q')
axes[0].set_title('qubit envelope')
axes[0].set_xlabel('time [us]')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(system['t_mid'], cavity_drive.real, label='I')
axes[1].plot(system['t_mid'], cavity_drive.imag, label='Q')
axes[1].set_title('cavity envelope before IQ convention')
axes[1].set_xlabel('time [us]')
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.show()
